# 02 â€” Feature Engineering V2

**Objetivo:** Transformar el dataset limpio en un dataset listo para modelado, preservando Ãºnicamente seÃ±ales Ãºtiles disponibles al momento de llegada del lead y aplicando tÃ©cnicas de codificaciÃ³n robustas para variables categÃ³ricas.

**Pasos de este notebook:**
1. Cargar el dataset limpio y preparar el entorno
2. Validar y eliminar variables redundantes si aÃºn existen
3. Crear nuevas features temporales derivadas
4. Agrupar categorÃ­as de baja frecuencia
5. Separar Train/Test antes del encoding para evitar data leakage
6. Aplicar One-Hot Encoding + variables numericas/derivadas y One-Hot Encoding selectivo
7. Exportar datasets finales para modelado

**Comparativa V1 vs V2:**
- **VersiÃ³n 1:** El encoding categÃ³rico utilizaba One-Hot Encoding masivo y una tÃ©cnica manual de "Encoding de contribuciÃ³n + codificacion smoothing" que creaba 2 columnas por variable para evitar el sesgo de las categorÃ­as raras.
- **VersiÃ³n 2 (Este notebook):** Se utiliza la librerÃ­a estÃ¡ndar `category_encoders.TargetEncoder`, que aplica codificacion consistente para produccion de forma robusta y reproducible. Esto evita la necesidad de crear 2 columnas por variable, simplifica el cÃ³digo, reduce la dimensionalidad y mantiene protecciÃ³n contra categorÃ­as de bajo volumen.

> Nota de version activa: el artifact desplegado en la app usa una matriz final compuesta por One-Hot + variables numericas/derivadas, sin columnas bayesianas materializadas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split

CLEAN_PATH = "../data/processed/leads_cleaned.csv"
df = pd.read_csv(CLEAN_PATH)

sns.set(style="whitegrid")
print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Columnas disponibles: {list(df.columns)}")

## 1. Validar y eliminar features sin valor predictivo

En este paso verificamos si aÃºn sobreviven columnas redundantes, de varianza nula o con riesgo de contaminar el modelado. Como el dataset ya viene de `00_data_engineering_v2_2025.ipynb`, aquÃ­ actuamos de forma defensiva: solo eliminamos columnas si todavÃ­a existen.

**Diferencias V1 vs V2:**
En la V1 parte de esta limpieza se resolvÃ­a mÃ¡s tarde y quedaba mezclada con el encoding. En la V2 dejamos este control explÃ­cito al inicio para que el pipeline sea mÃ¡s fÃ¡cil de auditar y mantener.

> Nota de version activa: el artifact desplegado en la app usa una matriz final compuesta por One-Hot + variables numericas/derivadas, sin columnas bayesianas materializadas.

In [ ]:
cols_eliminar = ["anio_creacion", "subtipo_interes", "plataforma"]
cols_presentes = [c for c in cols_eliminar if c in df.columns]

if cols_presentes:
    df = df.drop(columns=cols_presentes)
    print(f"Columnas eliminadas: {cols_presentes}")
else:
    print("No habÃ­a columnas redundantes adicionales por eliminar.")

print(f"Dataset tras validaciÃ³n inicial: {df.shape[0]:,} filas x {df.shape[1]} columnas")

## 2. Crear nuevas features derivadas

A partir de las variables temporales del dataset limpio, generamos seÃ±ales mÃ¡s interpretables para el modelo. En particular:
- `es_fin_de_semana`: captura diferencias de comportamiento entre dÃ­as hÃ¡biles y fines de semana.
- `franja_horaria`: resume la hora exacta en bloques comerciales mÃ¡s estables.

**Hallazgos esperados:**
Estas transformaciones suelen capturar mejor patrones operativos que la hora cruda, especialmente cuando el volumen de leads cambia por jornada.

**Diferencias V1 vs V2:**
En la V1 la seÃ±al horaria estaba degradada porque se derivaba desde una columna del sistema. En la V2 aprovechamos la hora real del cliente ya corregida en Data Engineering.

> Nota de version activa: el artifact desplegado en la app usa una matriz final compuesta por One-Hot + variables numericas/derivadas, sin columnas bayesianas materializadas.

In [ ]:
if "dia_semana_creacion" in df.columns:
    df["es_fin_de_semana"] = df["dia_semana_creacion"].isin(["sÃ¡bado", "domingo"]).astype(int)

def clasificar_franja(hora):
    if 0 <= hora < 6:
        return "madrugada"
    elif 6 <= hora < 12:
        return "manana"
    elif 12 <= hora < 18:
        return "tarde"
    return "noche"

if "hora_creacion" in df.columns:
    df["franja_horaria"] = df["hora_creacion"].apply(clasificar_franja)

print("Nuevas features derivadas creadas exitosamente.")

if {"franja_horaria", "target"}.issubset(df.columns):
    resumen_franja = (
        df.groupby("franja_horaria")["target"]
        .agg(["count", "mean"])
        .sort_values("count", ascending=False)
    )
    display(resumen_franja)

    plt.figure(figsize=(7, 4))
    ax = (resumen_franja["mean"] * 100).plot(kind="bar", color="#3498db", edgecolor="black")
    plt.title("Tasa de Hot Leads por franja horaria")
    plt.ylabel("Hot Leads (%)")
    plt.xlabel("Franja horaria")
    plt.xticks(rotation=0)
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=10)
    plt.tight_layout()
    plt.show()

## 3. Agrupar categorÃ­as de baja frecuencia

Las categorÃ­as extremadamente raras generan ruido y pueden producir estimaciones inestables durante el encoding. Para reducir ese efecto, agrupamos en `otros` las categorÃ­as con frecuencia menor al 1% del dataset.

**Hallazgos esperados:**
Este paso suele reducir dispersiÃ³n, mejorar robustez estadÃ­stica y evitar que el encoder sobreinterprete categorÃ­as con muy pocos ejemplos.

**Diferencias V1 vs V2:**
En la V1 parte de esta robustez se intentaba recuperar despuÃ©s mediante columnas manuales de contribuciÃ³n. En la V2 reducimos el problema desde antes, agrupando categorÃ­as raras y luego aplicando Target Encoding.

> Nota de version activa: el artifact desplegado en la app usa una matriz final compuesta por One-Hot + variables numericas/derivadas, sin columnas bayesianas materializadas.

In [ ]:
UMBRAL_PCT = 1.0
umbral_abs = int(len(df) * UMBRAL_PCT / 100)

cat_cols_agrupar = ["nombre_formulario", "vehiculo_interes", "origen", "campana", "concesionario"]
resumen_agrupacion = []

for col in cat_cols_agrupar:
    if col not in df.columns:
        continue

    vc = df[col].value_counts()
    bajas = vc[vc < umbral_abs].index
    filas_afectadas = int(df[col].isin(bajas).sum())

    if len(bajas) > 0:
        df.loc[df[col].isin(bajas), col] = "otros"

    resumen_agrupacion.append({
        "columna": col,
        "categorias_originales": int(vc.shape[0]),
        "categorias_agrupadas": int(len(bajas)),
        "filas_reetiquetadas": filas_afectadas,
        "categorias_finales": int(df[col].nunique()),
    })

resumen_agrupacion = pd.DataFrame(resumen_agrupacion)
display(resumen_agrupacion)

if not resumen_agrupacion.empty:
    plt.figure(figsize=(8, 4))
    ax = sns.barplot(data=resumen_agrupacion, x="columna", y="filas_reetiquetadas", color="#9b59b6")
    plt.title("Filas reetiquetadas como 'otros' por variable")
    plt.ylabel("NÃºmero de filas")
    plt.xlabel("Variable")
    plt.xticks(rotation=30, ha="right")
    for p in ax.patches:
        ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()

## 4. Split Train/Test

Es crÃ­tico separar el dataset **antes** de aplicar cualquier encoding para evitar **data leakage**. Si calculÃ¡ramos estadÃ­sticas sobre todo el dataset y luego dividiÃ©ramos, el modelo recibirÃ­a informaciÃ³n indirecta del conjunto de prueba.

**Hallazgo esperado:**
DespuÃ©s del split, la proporciÃ³n de Hot/Cold debe mantenerse casi igual en ambos conjuntos gracias a `stratify=y`.

**Diferencias V1 vs V2:**
En la V1 habÃ­a mÃ¡s transformaciones manuales previas al modelado. En la V2 hacemos explÃ­cita esta frontera para que cualquier transformaciÃ³n supervisada ocurra exclusivamente sobre train.

> Nota de version activa: el artifact desplegado en la app usa una matriz final compuesta por One-Hot + variables numericas/derivadas, sin columnas bayesianas materializadas.

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,} filas")
print(f"Test:  {X_test.shape[0]:,} filas")
print(f"Hot rate train: {y_train.mean()*100:.2f}%")
print(f"Hot rate test:  {y_test.mean()*100:.2f}%")

## 5. codificacion Target Encoding (Reemplaza la lÃ³gica manual de V1)

### 5.1 Â¿Por quÃ© ya no usamos el "Encoding de contribuciÃ³n" manual?
En la V1, el paso de contribuciÃ³n + codificacion smoothing creaba manualmente dos columnas por variable para evitar que categorÃ­as raras con tasas extremas confundieran al modelo.

En la V2 utilizamos **`category_encoders.TargetEncoder`** con el parÃ¡metro `smoothing`. Esta implementaciÃ³n aplica el mismo principio estadÃ­stico de suavizado codificaciono, pero de forma mÃ¡s compacta, robusta y mantenible.

**Hallazgos esperados:**
- Menor dimensionalidad que en V1
- Menor complejidad de mantenimiento
- Menor riesgo de errores manuales en el cÃ¡lculo de tasas suavizadas

**Diferencias V1 vs V2:**
V1 construÃ­a columnas derivadas manualmente. V2 delega esa responsabilidad en una librerÃ­a especializada y ampliamente usada, manteniendo el control del leakage al entrenarla solo sobre `X_train`.

> Nota de version activa: el artifact desplegado en la app usa una matriz final compuesta por One-Hot + variables numericas/derivadas, sin columnas bayesianas materializadas.

In [ ]:
# Variables categoricas de alta cardinalidad a codificar
encode_cols = ["vehiculo_interes", "origen", "nombre_formulario", "campana", "concesionario"]
encode_cols = [c for c in encode_cols if c in X_train.columns]

# TargetEncoder aplica la codificacion categorica definida en el notebook
encoder = TargetEncoder(cols=encode_cols, smoothing=100)

X_train_enc = encoder.fit_transform(X_train, y_train)
X_test_enc = encoder.transform(X_test)

print("Codificacion categorica aplicada con exito.")
print(f"Columnas con Target Encoding: {encode_cols}")

if encode_cols:
    preview_cols = encode_cols[: min(3, len(encode_cols))]
    display(X_train_enc[preview_cols].head())

### 5.2 One-Hot Encoding para variables de baja cardinalidad

Las variables categÃ³ricas restantes, generalmente de baja cardinalidad, se codifican con One-Hot Encoding. Este enfoque conserva interpretabilidad sin inflar innecesariamente el nÃºmero de columnas.

**Diferencias V1 vs V2:**
En la V1 el One-Hot tenÃ­a mucho mÃ¡s peso porque se aplicaba de forma mÃ¡s agresiva. En la V2 queda reservado para variables pequeÃ±as, mientras que las variables con mÃ¡s categorÃ­as pasan por Target Encoding.

> Nota de version activa: el artifact desplegado en la app usa una matriz final compuesta por One-Hot + variables numericas/derivadas, sin columnas bayesianas materializadas.

In [ ]:
onehot_cols = list(X_train_enc.select_dtypes(include=["object"]).columns)

X_train_final = pd.get_dummies(X_train_enc, columns=onehot_cols, drop_first=True, dtype=int)
X_test_final = pd.get_dummies(X_test_enc, columns=onehot_cols, drop_first=True, dtype=int)

# Alinear columnas
X_test_final = X_test_final.reindex(columns=X_train_final.columns, fill_value=0)

print(f"Columnas codificadas con One-Hot: {onehot_cols}")
print(f"Dimensiones finales Train: {X_train_final.shape}")
print(f"Dimensiones finales Test:  {X_test_final.shape}")
print(f"Total de features finales: {X_train_final.shape[1]}")

plt.figure(figsize=(6, 4))
resumen_dim = pd.Series({
    "Antes del encoding": X_train.shape[1],
    "DespuÃ©s del encoding": X_train_final.shape[1],
})
ax = resumen_dim.plot(kind="bar", color=["#95a5a6", "#2ecc71"], edgecolor="black")
plt.title("ComparaciÃ³n de dimensionalidad (train)")
plt.ylabel("NÃºmero de features")
plt.xticks(rotation=0)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.tight_layout()
plt.show()

**Hallazgos de la codificaciÃ³n V2:**
La estrategia hÃ­brida aplicada en este notebook logra un equilibrio Ãºtil entre densidad estadÃ­stica, interpretabilidad y control del sobreajuste. Las variables de alta cardinalidad quedan resumidas en una sola seÃ±al numÃ©rica suavizada, mientras que las variables pequeÃ±as conservan su estructura con One-Hot Encoding.

**Diferencias V1 vs V2:**
La V1 dependÃ­a mÃ¡s de ingenierÃ­a manual y generaba mÃ¡s columnas auxiliares. La V2 reduce complejidad, estandariza el pipeline y facilita que la inferencia en producciÃ³n replique exactamente el mismo comportamiento.

> Nota de version activa: el artifact desplegado en la app usa una matriz final compuesta por One-Hot + variables numericas/derivadas, sin columnas bayesianas materializadas.

## 6. Exportar datasets para modelado

Una vez finalizado el feature engineering, exportamos los conjuntos `X_train`, `X_test`, `y_train` y `y_test` para que el notebook de modelado pueda enfocarse Ãºnicamente en entrenamiento y evaluaciÃ³n.

**Diferencias V1 vs V2:**
En la V1 parte de estas transformaciones y validaciones quedaban mÃ¡s acopladas al modelado. En la V2 separamos responsabilidades: este notebook construye features; el siguiente entrena modelos.

> Nota de version activa: el artifact desplegado en la app usa una matriz final compuesta por One-Hot + variables numericas/derivadas, sin columnas bayesianas materializadas.

In [ ]:
import os

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_train_final.to_csv(f"{OUTPUT_DIR}/X_train_v2.csv", index=False)
X_test_final.to_csv(f"{OUTPUT_DIR}/X_test_v2.csv", index=False)
y_train.to_csv(f"{OUTPUT_DIR}/y_train_v2.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}/y_test_v2.csv", index=False)

print("Archivos exportados exitosamente.")
print(f"X_train_v2.csv: {X_train_final.shape}")
print(f"X_test_v2.csv:  {X_test_final.shape}")
print(f"y_train_v2.csv: {y_train.shape}")
print(f"y_test_v2.csv:  {y_test.shape}")